<a href="https://colab.research.google.com/github/YomnaEsmail/Masters/blob/main/Predicting_paper_Model_pCNN_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 👀About this Code
Simulate the paper model

LB = 24 and 48 historical time steps as paper

PL = 6 and PL=12  future time steps as paper [30 minutes and 1 hour]

Batch size = 128 as paper for both LP=6 & 12

Epochs = 120 and 165 as paper


43_cleaned_original.csv
          │
          ▼
   Remove Timestampms
   Remove CPUusageMHZ
          │
          ▼
  All remaining main
     input features
          │
          ▼
 Chronological 65/35 split
          │
          ▼
Scaler fitted ONLY on training data
          │
          ▼
     LB = 48
          │
          ▼
     pCNN-LSTM
          │
          ▼
  Predict t+1 ... t+12
          │
          ▼
    5 independent runs
          │
          ▼
 MSE / RMSE / MAE / R² / MAPE
          │
          ├── Overall metrics
          ├── Per-horizon metrics
          ├── Predictions
          ├── Training history
          ├── Model
          ├── Scalers
          ├── Sequences
          ├── Configuration
          └── Graphs


**Parallel CNN-LSTM (pCNN-LSTM) 1st trace: PL=6, LB=24, KS=3**

* **Description:** An advanced multi-branch architecture using stacked dilated convolutions to extract temporal patterns across different temporal depths, fed into a heavy LSTM layer.
* **Parameters & Specs:**
* **Input Layer:** Shape `(lookback, features)`
* **Branch 1 (Stacked):** Conv1D (64 filters, kernel size 3, dilation 1) $\rightarrow$ Conv1D (64 filters, kernel size 3, dilation 2)
* **Branch 2:** *(Commented out/Disabled)*
* **Branch 3 (Single-layer):** Conv1D (64 filters, kernel size 3, dilation 1)
* **Merging:** Concatenation layer combining output of Branch 1 and Branch 3
* **Recurrent & Output Layers:** LSTM (128 units, max-norm constraint 5.0), Output Dense (`pred_length` units, linear activation)
* **Compilation:** Adam optimizer (learning rate $0.05$, gradient norm clipping $5.0$), MSE loss, MAE metric

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU Successfully Activated: {gpus[0].name}")
else:
    print("GPU not detected. Make sure Runtime settings are saved.")

GPU Successfully Activated: /physical_device:GPU:0


In [ ]:
# Prevent TensorFlow from pre-allocating all VRAM at once
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
# =========================================================
# 1. IMPORTS
# =========================================================

import os
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import time
import random
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from tensorflow.keras.callbacks import LearningRateScheduler
from matplotlib.backends.backend_pdf import PdfPages

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from sklearn.feature_selection import (
    mutual_info_regression,
    RFE,
    SelectKBest,
    f_regression
)

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    LSTM,
    GRU,
    Conv1D,
    Dropout,
    Bidirectional,
    Input
)

from tensorflow.keras.constraints import max_norm
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import time

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    LSTM,
    Bidirectional,
    GRU,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Conv1D, Concatenate, LSTM, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.layers import Input, Conv1D, Concatenate, LSTM, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Bidirectional, LSTM, Dense, Dropout
warnings.filterwarnings("ignore")



**Loading Data & Feature Selection Preparation**

In [ ]:
# =========================================================
# 2. REPRODUCIBILITY
# =========================================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# =========================================================
# 3. STYLE
# =========================================================

sns.set_style("whitegrid")
sns.set_context("talk")

palette = sns.color_palette("Set2")

# =========================================================
# 4. LOAD DATA
# =========================================================

DATA_PATH = "43_cleaned_original.csv"
TARGET = "CPUusageMHZ"

df = pd.read_csv(DATA_PATH)

print("\nDataset Shape:", df.shape)

if TARGET not in df.columns:
    raise KeyError(f"Target column '{TARGET}' not found.")

print("\nColumns:")
print(df.columns.tolist())


Dataset Shape: (8632, 11)

Columns:
['Timestampms', 'CPUcores', 'CPUcapacityprovisionedMHZ', 'CPUusageMHZ', 'CPUusage%', 'MemorycapacityprovisionedKB', 'MemoryusageKB', 'DiskreadthroughputKBs', 'DiskwritethroughputKBs', 'NetworkreceivedthroughputKBs', 'NetworktransmittedthroughputKBs']


In [ ]:
# =========================================================
# MAIN INPUT FEATURES
# =========================================================

# Exclude the prediction target
# Exclude Timestampms because it is an index/time identifier,
# not a workload feature.

EXCLUDED_COLUMNS = [
    TARGET,
    "Timestampms"
]

features = [
    col
    for col in df.columns
    if col not in EXCLUDED_COLUMNS
]

# Keep only numeric features
features = [
    col
    for col in features
    if pd.api.types.is_numeric_dtype(df[col])
]


if len(features) == 0:

    raise ValueError(
        "No numeric input features were found."
    )


print("\n" + "=" * 70)
print("MAIN INPUT FEATURES")
print("=" * 70)

print(
    f"Number of input features: {len(features)}"
)

for i, feature in enumerate(features, 1):

    print(
        f"{i}. {feature}"
    )


MAIN INPUT FEATURES
Number of input features: 9
1. CPUcores
2. CPUcapacityprovisionedMHZ
3. CPUusage%
4. MemorycapacityprovisionedKB
5. MemoryusageKB
6. DiskreadthroughputKBs
7. DiskwritethroughputKBs
8. NetworkreceivedthroughputKBs
9. NetworktransmittedthroughputKBs


In [ ]:
# =========================================================
# MULTI-STEP FORECASTING CONFIGURATION
# =========================================================

LB = 48       #24           # Lookback window
PL = 12        #6           # Prediction horizon
BATCH_SIZE = 128
EPOCHS = 165 #120
N_RUNS = 5

TRAIN_RATIO = 0.65
TEST_RATIO = 1 - TRAIN_RATIO
print("\n======================================================")
print("MULTI-STEP FORECASTING CONFIGURATION")
print("======================================================")
print(f"Lookback Window (LB): {LB}")
print(f"Prediction Length (PL): {PL}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Runs: {N_RUNS}")
print(f"Train/Test Split: {TRAIN_RATIO:.0%}/{1-TRAIN_RATIO:.0%}")


MULTI-STEP FORECASTING CONFIGURATION
Lookback Window (LB): 24
Prediction Length (PL): 6
Batch Size: 128
Epochs: 120
Runs: 5
Train/Test Split: 65%/35%


In [ ]:
# =========================================================
# MULTI-STEP SEQUENCE CREATION
# =========================================================

def create_multistep_sequences(
    X,
    y,
    lookback,
    pred_length
):

    X_seq = []
    y_seq = []

    for i in range(
        lookback,
        len(X) - pred_length + 1
    ):

        X_seq.append(
            X[i-lookback:i]
        )

        y_seq.append(
            y[i:i+pred_length]
        )

    return (
        np.array(X_seq),
        np.array(y_seq)
    )

In [ ]:

# =========================================================
# M6: pCNN-LSTM - PAPER ARCHITECTURE  CB1 & CB2 two layers
# LB=24 / PL=6
# =========================================================

def build_paper_pCNNLSTM(
    input_shape,
    pred_length=12
):

    input_seq = Input(
        shape=input_shape
    )

    # -----------------------------------------------------
    # Branch 1
    # KS=3, DL=1
    # followed by KS=3, DL=2
    # -----------------------------------------------------

    cb1_l1 = Conv1D(filters=64,kernel_size=3,dilation_rate=1,padding="causal",activation="relu",kernel_constraint=max_norm(5.0))(input_seq)

    cb1_l2 = Conv1D(filters=64,kernel_size=3,dilation_rate=2,padding="causal",activation="relu",kernel_constraint=max_norm(5.0))(cb1_l1)

    # -----------------------------------------------------
    # Branch 2 used with PL=12
    # KS=3, DL=1
    # followed by KS=6, DL=2
    # -----------------------------------------------------

    cb2_l1 = Conv1D(filters=64,kernel_size=3,dilation_rate=1,padding="causal",activation="relu",kernel_constraint=max_norm(5.0))(input_seq)

    cb2_l2 = Conv1D(filters=64,kernel_size=6,dilation_rate=2,padding="causal",activation="relu",kernel_constraint=max_norm(5.0))(cb2_l1)

    # -----------------------------------------------------
    # Branch 3
    # KS=3, DL=1
    # -----------------------------------------------------

    cb3_l1 = Conv1D(filters=64,kernel_size=3,dilation_rate=1,padding="causal",activation="relu",kernel_constraint=max_norm(5.0))(input_seq)

    # -----------------------------------------------------
    # Merge CNN branches
    # -----------------------------------------------------

    merged = Concatenate()([
        cb1_l2,
        cb2_l2, #used with PL=12
        cb3_l1
    ]) #cb2_l2, removed

    # -----------------------------------------------------
    # LSTM
    # -----------------------------------------------------

    lstm_out =LSTM(128,kernel_constraint=max_norm(5.0))(merged)

    # -----------------------------------------------------
    # Multi-step output
    # -----------------------------------------------------

    output = Dense(
        pred_length,
        activation="linear"
    )(lstm_out)

    model = Model(
        inputs=input_seq,
        outputs=output
    )

    # -----------------------------------------------------
    # Optimizer
    # -----------------------------------------------------

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=0.05,
        clipnorm=5.0
    )

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    return model


In [ ]:
# =========================================================
# LEARNING RATE DECAY same as pCNN-LSTM Paper parameters
# =========================================================

def step_decay(epoch):

    initial_lr = 0.05
    drop_rate = 0.5
    epochs_drop = 30

    return float(
        initial_lr *
        (
            drop_rate **
            (epoch // epochs_drop)
        )
    )

In [ ]:
# =========================================================
# MODEL LIST
# =========================================================

models = [


    (
        "Model 1: pCNN-LSTM",
        build_paper_pCNNLSTM,
        "paper"
    )

]

In [ ]:
# =========================================================
# GOOGLE DRIVE
# =========================================================

import os
import json
import time
import random
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from google.colab import drive

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from tensorflow.keras.callbacks import LearningRateScheduler


# =========================================================
# MOUNT GOOGLE DRIVE
# =========================================================

drive.mount('/content/drive')


# =========================================================
# BASE DIRECTORY
# =========================================================
# Everything will be saved under:
# Google Drive > MyDrive > Time_Series_Experiments > PL12_Results

#BASE_DIR = "/content/drive/MyDrive/Time_Series_Experiments/PL12_Results"

# Google Drive root for this project
PROJECT_DIR = "/content/drive/MyDrive/Time_Series_Experiments"

# Results for this particular experiment/model
BASE_DIR = os.path.join(PROJECT_DIR, "Predicting_BaseModels_pCNN_LSTM[PL=12, 5RUNs]")
# =========================================================
# OUTPUT DIRECTORIES
# =========================================================

DIR_PREDICTIONS = os.path.join(
    BASE_DIR,
    "Predictions"
)

DIR_MODELS = os.path.join(
    BASE_DIR,
    "Saved_Models"
)

DIR_HISTORY = os.path.join(
    BASE_DIR,
    "Training_History"
)

DIR_METRICS = os.path.join(
    BASE_DIR,
    "Metrics"
)

DIR_GRAPHS = os.path.join(
    BASE_DIR,
    "Graphs"
)

DIR_PER_HORIZON = os.path.join(
    BASE_DIR,
    "Per_Horizon_Metrics"
)

DIR_SCALERS = os.path.join(
    BASE_DIR,
    "Scalers"
)

DIR_SEQUENCES = os.path.join(
    BASE_DIR,
    "Sequences"
)

DIR_CONFIG = os.path.join(
    BASE_DIR,
    "Configurations"
)

DIR_SUMMARY = os.path.join(
    BASE_DIR,
    "Model_Summaries"
)


# =========================================================
# CREATE DIRECTORIES
# =========================================================

ALL_DIRS = [
    BASE_DIR,
    DIR_PREDICTIONS,
    DIR_MODELS,
    DIR_HISTORY,
    DIR_METRICS,
    DIR_GRAPHS,
    DIR_PER_HORIZON,
    DIR_SCALERS,
    DIR_SEQUENCES,
    DIR_CONFIG,
    DIR_SUMMARY
]

for directory in ALL_DIRS:
    os.makedirs(directory, exist_ok=True)


# =========================================================
# VERIFY
# =========================================================

print("All output directories are ready.")
print()
print("Base directory:")
print(BASE_DIR)
print()

for directory in ALL_DIRS:
    print(directory)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All output directories are ready.

Base directory:
/content/drive/MyDrive/Time_Series_Experiments/Predicting_BaseModels_pCNN_LSTM[PL=6, 5RUNs]

/content/drive/MyDrive/Time_Series_Experiments/Predicting_BaseModels_pCNN_LSTM[PL=6, 5RUNs]
/content/drive/MyDrive/Time_Series_Experiments/Predicting_BaseModels_pCNN_LSTM[PL=6, 5RUNs]/Predictions
/content/drive/MyDrive/Time_Series_Experiments/Predicting_BaseModels_pCNN_LSTM[PL=6, 5RUNs]/Saved_Models
/content/drive/MyDrive/Time_Series_Experiments/Predicting_BaseModels_pCNN_LSTM[PL=6, 5RUNs]/Training_History
/content/drive/MyDrive/Time_Series_Experiments/Predicting_BaseModels_pCNN_LSTM[PL=6, 5RUNs]/Metrics
/content/drive/MyDrive/Time_Series_Experiments/Predicting_BaseModels_pCNN_LSTM[PL=6, 5RUNs]/Graphs
/content/drive/MyDrive/Time_Series_Experiments/Predicting_BaseModels_pCNN_LSTM[PL=6, 5RUNs]/Per_Horizon_Metrics
/conte

In [ ]:
# =========================================================
# SAFE FILE NAME
# =========================================================

def make_safe_name(text):

    return (
        str(text)
        .replace(":", "")
        .replace(" ", "_")
        .replace("-", "")
        .replace("/", "_")
        .replace("\\", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("[", "")
        .replace("]", "")
        .replace("%", "pct")
    )

In [ ]:
# =========================================================
# MAPE FUNCTION
# =========================================================

def calculate_mape(
    y_true,
    y_pred
):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    non_zero = y_true != 0

    if not np.any(non_zero):
        return np.nan

    return (
        np.mean(
            np.abs(
                (
                    y_true[non_zero]
                    -
                    y_pred[non_zero]
                )
                /
                y_true[non_zero]
            )
        )
        * 100
    )

In [ ]:
# =========================================================
# SAVE MODEL SUMMARY
# =========================================================

def save_model_summary(
    model,
    path
):

    with open(
        path,
        "w"
    ) as f:

        model.summary(
            print_fn=lambda x: f.write(
                x + "\n"
            )
        )

In [ ]:
# =========================================================
# SAVE TRAINING LOSS GRAPH
# =========================================================

def save_training_graph(
    history,
    prefix
):

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        history.history["loss"],
        label="Training Loss"
    )

    if "val_loss" in history.history:

        plt.plot(
            history.history["val_loss"],
            label="Validation Loss"
        )

    plt.xlabel(
        "Epoch"
    )

    plt.ylabel(
        "Loss"
    )

    plt.title(
        f"Training and Validation Loss - {prefix}"
    )

    plt.legend()

    plt.tight_layout()

    path = os.path.join(
        DIR_GRAPHS,
        prefix + "_TrainingLoss.png"
    )

    plt.savefig(
        path,
        dpi=300
    )

    plt.close()

    return path

In [ ]:
# =========================================================
# SAVE ACTUAL VS PREDICTED GRAPH
# =========================================================

def save_prediction_graphs(
    y_true,
    y_pred,
    prefix
):

    graph_paths = []

    horizons = [
        0,
        y_true.shape[1] - 1
    ]

    for h in horizons:

        plt.figure(
            figsize=(12, 5)
        )

        plt.plot(
            y_true[:, h],
            label=f"Actual t+{h+1}"
        )

        plt.plot(
            y_pred[:, h],
            label=f"Predicted t+{h+1}"
        )

        plt.xlabel(
            "Test Sequence"
        )

        plt.ylabel(
            TARGET
        )

        plt.title(
            f"Actual vs Predicted - "
            f"t+{h+1} - {prefix}"
        )

        plt.legend()

        plt.tight_layout()

        path = os.path.join(
            DIR_GRAPHS,
            prefix +
            f"_Actual_vs_Predicted_t+{h+1}.png"
        )

        plt.savefig(
            path,
            dpi=300
        )

        plt.close()

        graph_paths.append(
            path
        )

    return graph_paths

In [ ]:
# =========================================================
# STORAGE
# =========================================================

results = []
horizon_results = []

predictions = {}
histories = {}

In [ ]:
# =========================================================
# EXPERIMENTAL RESULTS
# NO FEATURE SELECTION
# ALL MAIN INPUT FEATURES
# =========================================================


# =========================================================
# RESULT FILES
# =========================================================

MAIN_RESULTS_FILE = os.path.join(
    DIR_METRICS,
    "ALL_RUN_RESULTS_PL12.csv"
)

HORIZON_RESULTS_FILE = os.path.join(
    DIR_PER_HORIZON,
    "ALL_HORIZON_RESULTS_PL12.csv"
)


# =========================================================
# STORAGE
# =========================================================

results = []
horizon_results = []

predictions = {}
histories = {}


# =========================================================
# LOAD EXISTING RESULTS
# =========================================================

if os.path.exists(
    MAIN_RESULTS_FILE
):

    existing_results = pd.read_csv(
        MAIN_RESULTS_FILE
    )

    results = (
        existing_results
        .to_dict(
            orient="records"
        )
    )

    print(
        "Loaded existing main results:",
        len(results)
    )


if os.path.exists(
    HORIZON_RESULTS_FILE
):

    existing_horizon_results = pd.read_csv(
        HORIZON_RESULTS_FILE
    )

    horizon_results = (
        existing_horizon_results
        .to_dict(
            orient="records"
        )
    )

    print(
        "Loaded existing horizon results:",
        len(horizon_results)
    )


# =========================================================
# COMPLETED RUN IDENTIFIERS
# =========================================================
#
# There is NO feature-selection dimension anymore.
#
# Identifier:
#
#       Model + Run
#
# =========================================================

completed_runs = set()


for row in results:

    completed_runs.add(
        (
            row["Model"],
            int(row["Run"])
        )
    )


print(
    "Previously completed runs:",
    len(completed_runs)
)


# =========================================================
# MAIN EXPERIMENT
# =========================================================

for model_name, model_fn, model_type in models:

    print("\n" + "=" * 80)
    print(model_name)
    print("=" * 80)


    # =====================================================
    # RUNS
    # =====================================================

    for run in range(
        1,
        N_RUNS + 1
    ):


        # =================================================
        # RUN IDENTIFIER
        # =================================================

        run_identifier = (
            model_name,
            run
        )


        # =================================================
        # SKIP ALREADY COMPLETED RUN
        # =================================================

        if run_identifier in completed_runs:

            print(
                f"SKIPPING existing run: "
                f"{model_name} | "
                f"Run {run}"
            )

            continue


        print(
            "\n" +
            "-" * 80
        )

        print(
            f"{model_name} | "
            f"Run {run}/{N_RUNS}"
        )

        print(
            "-" * 80
        )


        # =================================================
        # SAFE MODEL NAME
        # =================================================

        safe_model = make_safe_name(
            model_name
        )


        # =================================================
        # FILE PREFIX
        # =================================================

        prefix = (
            f"{safe_model}_"
            f"AllFeatures_"
            f"LB{LB}_"
            f"PL{PL}_"
            f"Run{run}"
        )


        # =================================================
        # REPRODUCIBILITY
        # =================================================

        run_seed = (
            SEED + run
        )

        np.random.seed(
            run_seed
        )

        random.seed(
            run_seed
        )

        tf.random.set_seed(
            run_seed
        )


        # =================================================
        # RAW DATA
        # =================================================

        X_raw = (
            df[features]
            .values
            .astype(np.float32)
        )

        y_raw = (
            df[[TARGET]]
            .values
            .astype(np.float32)
        )


        n_samples = len(df)


        # =================================================
        # CHRONOLOGICAL TRAIN / TEST SPLIT
        # =================================================

        raw_split = int(
            TRAIN_RATIO *
            n_samples
        )


        X_train_raw = (
            X_raw[:raw_split]
        )

        X_test_raw = (
            X_raw[raw_split:]
        )


        y_train_raw = (
            y_raw[:raw_split]
        )

        y_test_raw = (
            y_raw[raw_split:]
        )


        print(
            "Raw training samples:",
            len(X_train_raw)
        )

        print(
            "Raw testing samples:",
            len(X_test_raw)
        )


        # =================================================
        # FIT SCALERS ONLY ON TRAINING DATA
        # =================================================

        scaler_X = MinMaxScaler()

        scaler_y = MinMaxScaler()


        scaler_X.fit(
            X_train_raw
        )

        scaler_y.fit(
            y_train_raw
        )


        # =================================================
        # TRANSFORM TRAIN / TEST
        # =================================================

        X_train_scaled = (
            scaler_X.transform(
                X_train_raw
            )
        )

        X_test_scaled = (
            scaler_X.transform(
                X_test_raw
            )
        )


        y_train_scaled = (
            scaler_y.transform(
                y_train_raw
            )
        )

        y_test_scaled = (
            scaler_y.transform(
                y_test_raw
            )
        )


        # =================================================
        # CREATE TRAINING SEQUENCES
        # =================================================

        X_train_seq, y_train_seq = (
            create_multistep_sequences(
                X_train_scaled,
                y_train_scaled,
                lookback=LB,
                pred_length=PL
            )
        )


        # =================================================
        # CREATE TEST SEQUENCES
        #
        # Include LB training observations before the test
        # period so the first test prediction has history.
        # =================================================

        X_test_with_history = np.concatenate(
            [
                X_train_scaled[-LB:],
                X_test_scaled
            ],
            axis=0
        )


        y_test_with_history = np.concatenate(
            [
                y_train_scaled[-LB:],
                y_test_scaled
            ],
            axis=0
        )


        X_test_seq, y_test_seq = (
            create_multistep_sequences(
                X_test_with_history,
                y_test_with_history,
                lookback=LB,
                pred_length=PL
            )
        )


        # =================================================
        # REMOVE TARGETS EXTENDING BEYOND TEST PERIOD
        # =================================================

        expected_test_sequences = (
            len(y_test_raw)
            - PL
            + 1
        )


        X_test_seq = (
            X_test_seq[
                :expected_test_sequences
            ]
        )

        y_test_seq = (
            y_test_seq[
                :expected_test_sequences
            ]
        )


        print(
            "X_train:",
            X_train_seq.shape
        )

        print(
            "y_train:",
            y_train_seq.shape
        )

        print(
            "X_test:",
            X_test_seq.shape
        )

        print(
            "y_test:",
            y_test_seq.shape
        )


        # =================================================
        # MODEL TARGET
        # =================================================

        y_train_model = (
            y_train_seq.reshape(
                y_train_seq.shape[0],
                PL
            )
        )


        y_test_model = (
            y_test_seq.reshape(
                y_test_seq.shape[0],
                PL
            )
        )


        # =================================================
        # BUILD MODEL
        # =================================================
        #
        # IMPORTANT:
        # Pass PL explicitly.
        #
        # This guarantees output = (None, 12)
        # =================================================

        model = model_fn(
            (
                X_train_seq.shape[1],
                X_train_seq.shape[2]
            ),
            pred_length=PL
        )


        # =================================================
        # CHECK MODEL OUTPUT
        # =================================================

        output_shape = (
            model.output_shape
        )


        if output_shape[-1] != PL:

            raise ValueError(
                f"{model_name} output shape "
                f"{output_shape} does not match "
                f"PL={PL}"
            )


        print(
            "Model output:",
            output_shape
        )


        # =================================================
        # SAVE MODEL SUMMARY
        # =================================================

        summary_path = os.path.join(
            DIR_SUMMARY,
            prefix +
            "_ModelSummary.txt"
        )


        save_model_summary(
            model,
            summary_path
        )


        # =================================================
        # CALLBACKS
        # =================================================

        callbacks = []


        if model_type == "paper":

            callbacks.append(
                LearningRateScheduler(
                    step_decay,
                    verbose=0
                )
            )


        # =================================================
        # TRAINING
        # =================================================

        train_start = time.time()


        history = model.fit(

            X_train_seq,

            y_train_model,

            validation_split=0.10,

            epochs=EPOCHS,

            batch_size=BATCH_SIZE,

            callbacks=callbacks,

            verbose=0
        )


        training_time = (
            time.time()
            -
            train_start
        )


        print(
            f"Training time: "
            f"{training_time:.2f} sec"
        )


        # =================================================
        # PREDICTION
        # =================================================

        pred_start = time.time()


        y_pred_scaled = (
            model.predict(
                X_test_seq,
                verbose=0
            )
        )


        prediction_time = (
            time.time()
            -
            pred_start
        )


        # =================================================
        # INVERSE TRANSFORM
        # =================================================

        y_pred_inv = (
            scaler_y.inverse_transform(
                y_pred_scaled
            )
        )


        y_true_inv = (
            scaler_y.inverse_transform(
                y_test_model
            )
        )


        # =================================================
        # SAVE PREDICTIONS IMMEDIATELY
        # =================================================

        pred_data = {

            "TestSequence":
                np.arange(
                    len(y_true_inv)
                )
        }


        for h in range(PL):

            pred_data[
                f"Actual_t+{h+1}"
            ] = y_true_inv[:, h]

            pred_data[
                f"Predicted_t+{h+1}"
            ] = y_pred_inv[:, h]

            pred_data[
                f"Error_t+{h+1}"
            ] = (
                y_true_inv[:, h]
                -
                y_pred_inv[:, h]
            )

            pred_data[
                f"AbsoluteError_t+{h+1}"
            ] = np.abs(
                y_true_inv[:, h]
                -
                y_pred_inv[:, h]
            )


        pred_df = pd.DataFrame(
            pred_data
        )


        prediction_path = os.path.join(
            DIR_PREDICTIONS,
            prefix +
            "_Predictions.csv"
        )


        pred_df.to_csv(
            prediction_path,
            index=False
        )


        print(
            "Predictions saved."
        )


        # =================================================
        # SAVE TRAINING HISTORY
        # =================================================

        history_df = pd.DataFrame(
            history.history
        )


        history_path = os.path.join(
            DIR_HISTORY,
            prefix +
            "_History.csv"
        )


        history_df.to_csv(
            history_path,
            index=False
        )


        # =================================================
        # SAVE MODEL
        # =================================================

        model_path = os.path.join(
            DIR_MODELS,
            prefix +
            ".keras"
        )


        model.save(
            model_path
        )


        # =================================================
        # SAVE SCALERS
        # =================================================

        joblib.dump(
            scaler_X,
            os.path.join(
                DIR_SCALERS,
                prefix +
                "_ScalerX.pkl"
            )
        )


        joblib.dump(
            scaler_y,
            os.path.join(
                DIR_SCALERS,
                prefix +
                "_ScalerY.pkl"
            )
        )


        # =================================================
        # SAVE SEQUENCES
        # =================================================

        sequence_path = os.path.join(
            DIR_SEQUENCES,
            prefix +
            "_Sequences.npz"
        )


        np.savez_compressed(

            sequence_path,

            X_train_seq=X_train_seq,

            y_train_seq=y_train_seq,

            X_test_seq=X_test_seq,

            y_test_seq=y_test_seq,

            X_train_scaled=X_train_scaled,

            X_test_scaled=X_test_scaled,

            y_train_scaled=y_train_scaled,

            y_test_scaled=y_test_scaled
        )


        # =================================================
        # SAVE TEST DATA + PREDICTIONS
        # =================================================

        n_seq = len(
            y_pred_inv
        )


        test_export = pd.DataFrame({

            "TestSequence":
                np.arange(n_seq),

            "OriginalTestIndex":
                np.arange(
                    raw_split,
                    raw_split + n_seq
                )
        })


        # -------------------------------------------------
        # ORIGINAL TEST FEATURES
        # -------------------------------------------------

        for i, feature in enumerate(features):

            test_export[
                feature
            ] = X_test_raw[
                :n_seq,
                i
            ]


        # -------------------------------------------------
        # ACTUAL / PREDICTED / ERRORS
        # -------------------------------------------------

        for h in range(PL):

            test_export[
                f"Actual_t+{h+1}"
            ] = y_true_inv[:, h]

            test_export[
                f"Predicted_t+{h+1}"
            ] = y_pred_inv[:, h]

            test_export[
                f"Error_t+{h+1}"
            ] = (
                y_true_inv[:, h]
                -
                y_pred_inv[:, h]
            )

            test_export[
                f"AbsoluteError_t+{h+1}"
            ] = np.abs(
                y_true_inv[:, h]
                -
                y_pred_inv[:, h]
            )


        test_path = os.path.join(
            DIR_SEQUENCES,
            prefix +
            "_TestData_Predictions.csv"
        )


        test_export.to_csv(
            test_path,
            index=False
        )


        # =================================================
        # OVERALL METRICS
        # =================================================

        y_true_flat = (
            y_true_inv.flatten()
        )

        y_pred_flat = (
            y_pred_inv.flatten()
        )


        mse = mean_squared_error(
            y_true_flat,
            y_pred_flat
        )


        rmse = np.sqrt(
            mse
        )


        mae = mean_absolute_error(
            y_true_flat,
            y_pred_flat
        )


        r2 = r2_score(
            y_true_flat,
            y_pred_flat
        )


        mape = calculate_mape(
            y_true_flat,
            y_pred_flat
        )


        # =================================================
        # PER-HORIZON METRICS
        # =================================================

        current_horizon_results = []


        for h in range(PL):

            actual_h = (
                y_true_inv[:, h]
            )

            pred_h = (
                y_pred_inv[:, h]
            )


            mse_h = mean_squared_error(
                actual_h,
                pred_h
            )


            rmse_h = np.sqrt(
                mse_h
            )


            mae_h = mean_absolute_error(
                actual_h,
                pred_h
            )


            r2_h = r2_score(
                actual_h,
                pred_h
            )


            mape_h = calculate_mape(
                actual_h,
                pred_h
            )


            horizon_row = {

                "Model":
                    model_name,

                "Run":
                    run,

                "Lookback":
                    LB,

                "Prediction Length":
                    PL,

                "Horizon":
                    h + 1,

                "MSE":
                    mse_h,

                "RMSE":
                    rmse_h,

                "MAE":
                    mae_h,

                "R2":
                    r2_h,

                "MAPE":
                    mape_h
            }


            horizon_results.append(
                horizon_row
            )

            current_horizon_results.append(
                horizon_row
            )


        # =================================================
        # MAIN RESULT
        # =================================================

        result_row = {

            "Model":
                model_name,

            "Feature Selection":
                "None - All Main Features",

            "Selected Features":
                ", ".join(
                    features
                ),

            "Run":
                run,

            "Lookback":
                LB,

            "Prediction Length":
                PL,

            "Train Ratio":
                TRAIN_RATIO,

            "Test Ratio":
                TEST_RATIO,

            "Raw Training Samples":
                len(X_train_raw),

            "Raw Testing Samples":
                len(X_test_raw),

            "Training Sequences":
                len(X_train_seq),

            "Testing Sequences":
                len(X_test_seq),

            "Number of Features":
                len(features),

            "Epochs Used":
                len(
                    history.history[
                        "loss"
                    ]
                ),

            "MSE":
                mse,

            "RMSE":
                rmse,

            "MAE":
                mae,

            "R2":
                r2,

            "MAPE":
                mape,

            "Training Time (sec)":
                training_time,

            "Prediction Time (sec)":
                prediction_time,

            "Run Seed":
                run_seed
        }


        results.append(
            result_row
        )


        # =================================================
        # SAVE MAIN RESULTS IMMEDIATELY
        # =================================================

        pd.DataFrame(
            results
        ).to_csv(
            MAIN_RESULTS_FILE,
            index=False
        )


        # =================================================
        # SAVE HORIZON RESULTS IMMEDIATELY
        # =================================================

        pd.DataFrame(
            horizon_results
        ).to_csv(
            HORIZON_RESULTS_FILE,
            index=False
        )


        # =================================================
        # SAVE CONFIGURATION
        # =================================================

        config = {

            "Model":
                model_name,

            "Model Type":
                model_type,

            "Feature Selection":
                "None",

            "Feature Selection Method":
                None,

            "Selected Features":
                features,

            "Number of Features":
                len(features),

            "Run":
                run,

            "Lookback":
                LB,

            "Prediction Length":
                PL,

            "Batch Size":
                BATCH_SIZE,

            "Maximum Epochs":
                EPOCHS,

            "Actual Epochs Used":
                len(
                    history.history[
                        "loss"
                    ]
                ),

            "Train Ratio":
                TRAIN_RATIO,

            "Test Ratio":
                TEST_RATIO,

            "Target":
                TARGET,

            "Excluded Columns":
                EXCLUDED_COLUMNS,

            "Total Raw Samples":
                len(df),

            "Raw Training Samples":
                len(X_train_raw),

            "Raw Testing Samples":
                len(X_test_raw),

            "Training Sequences":
                len(X_train_seq),

            "Testing Sequences":
                len(X_test_seq),

            "Input Sequence Shape":
                list(
                    X_train_seq.shape
                ),

            "Target Sequence Shape":
                list(
                    y_train_seq.shape
                ),

            "Model Output Shape":
                list(
                    model.output_shape
                ),

            "MSE":
                mse,

            "RMSE":
                rmse,

            "MAE":
                mae,

            "R2":
                r2,

            "MAPE":
                mape,

            "Training Time (sec)":
                training_time,

            "Prediction Time (sec)":
                prediction_time,

            "Global Seed":
                SEED,

            "Run Seed":
                run_seed
        }


        config_path = os.path.join(
            DIR_CONFIG,
            prefix +
            "_Config.json"
        )


        with open(
            config_path,
            "w"
        ) as f:

            json.dump(
                config,
                f,
                indent=4,
                default=str
            )


        # =================================================
        # SAVE TRAINING GRAPH
        # =================================================

        save_training_graph(
            history,
            prefix
        )


        # =================================================
        # SAVE ACTUAL VS PREDICTED GRAPHS
        # =================================================

        save_prediction_graphs(
            y_true_inv,
            y_pred_inv,
            prefix
        )


        # =================================================
        # STORE IN MEMORY
        # =================================================

        predictions[
            (
                model_name,
                run
            )
        ] = (
            y_true_inv,
            y_pred_inv
        )


        histories[
            (
                model_name,
                run
            )
        ] = history


        # =================================================
        # MARK RUN AS COMPLETED
        # =================================================

        completed_runs.add(
            run_identifier
        )


        # =================================================
        # PRINT RESULTS
        # =================================================

        print(
            "\nRUN COMPLETED"
        )

        print(
            f"MSE   = {mse:.6f}"
        )

        print(
            f"RMSE  = {rmse:.6f}"
        )

        print(
            f"MAE   = {mae:.6f}"
        )

        print(
            f"R²    = {r2:.6f}"
        )

        print(
            f"MAPE  = {mape:.4f}%"
        )

        print(
            f"Training Time = "
            f"{training_time:.2f}s"
        )

        print(
            f"Prediction Time = "
            f"{prediction_time:.4f}s"
        )

        print(
            "All run data saved successfully."
        )

Previously completed runs: 0

Model 1: pCNN-LSTM

--------------------------------------------------------------------------------
Model 1: pCNN-LSTM | Run 1/5
--------------------------------------------------------------------------------
Raw training samples: 5610
Raw testing samples: 3022
X_train: (5581, 24, 9)
y_train: (5581, 6, 1)
X_test: (3017, 24, 9)
y_test: (3017, 6, 1)
Model output: (None, 6)


Training time: 55.69 sec
Predictions saved.

RUN COMPLETED
MSE   = 11418.992188
RMSE  = 106.859685
MAE   = 22.567617
R²    = -0.147016
MAPE  = 27.1998%
Training Time = 55.69s
Prediction Time = 0.5769s
All run data saved successfully.

--------------------------------------------------------------------------------
Model 1: pCNN-LSTM | Run 2/5
--------------------------------------------------------------------------------
Raw training samples: 5610
Raw testing samples: 3022
X_train: (5581, 24, 9)
y_train: (5581, 6, 1)
X_test: (3017, 24, 9)
y_test: (3017, 6, 1)
Model output: (None, 6)


Training time: 56.66 sec
Predictions saved.

RUN COMPLETED
MSE   = 12031.357422
RMSE  = 109.687545
MAE   = 22.773073
R²    = -0.208527
MAPE  = 29.4172%
Training Time = 56.66s
Prediction Time = 0.6033s
All run data saved successfully.

--------------------------------------------------------------------------------
Model 1: pCNN-LSTM | Run 3/5
--------------------------------------------------------------------------------
Raw training samples: 5610
Raw testing samples: 3022
X_train: (5581, 24, 9)
y_train: (5581, 6, 1)
X_test: (3017, 24, 9)
y_test: (3017, 6, 1)
Model output: (None, 6)


Training time: 55.76 sec
Predictions saved.

RUN COMPLETED
MSE   = 11439.078125
RMSE  = 106.953626
MAE   = 19.337759
R²    = -0.149033
MAPE  = 23.0556%
Training Time = 55.76s
Prediction Time = 0.5822s
All run data saved successfully.

--------------------------------------------------------------------------------
Model 1: pCNN-LSTM | Run 4/5
--------------------------------------------------------------------------------
Raw training samples: 5610
Raw testing samples: 3022
X_train: (5581, 24, 9)
y_train: (5581, 6, 1)
X_test: (3017, 24, 9)
y_test: (3017, 6, 1)
Model output: (None, 6)


Training time: 56.52 sec
Predictions saved.

RUN COMPLETED
MSE   = 12357.816406
RMSE  = 111.165716
MAE   = 22.464903
R²    = -0.241319
MAPE  = 26.6257%
Training Time = 56.52s
Prediction Time = 0.5900s
All run data saved successfully.

--------------------------------------------------------------------------------
Model 1: pCNN-LSTM | Run 5/5
--------------------------------------------------------------------------------
Raw training samples: 5610
Raw testing samples: 3022
X_train: (5581, 24, 9)
y_train: (5581, 6, 1)
X_test: (3017, 24, 9)
y_test: (3017, 6, 1)
Model output: (None, 6)


Training time: 55.20 sec
Predictions saved.

RUN COMPLETED
MSE   = 11745.959961
RMSE  = 108.378780
MAE   = 27.303476
R²    = -0.179859
MAPE  = 36.5504%
Training Time = 55.20s
Prediction Time = 0.5783s
All run data saved successfully.




***First, reconnect to my saved results:***